# 11 — Outliers con IQR estratificado
Regla de Tukey dentro de cada estrato materia × ámbito: un valor es outlier leve si supera
Q3 + 1,5·IQR y severo si supera Q3 + 3·IQR. **No se elimina ninguna fila**: se agregan banderas,
versiones winsorizadas (tope en el percentil 95 del estrato) y log(1 + x) de los volúmenes.

In [ ]:
%run -i modulos/comun.ipynb
%run -i modulos/outliers.ipynb
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

OUT_FEATURES = CURATED / "features"
OUT_EDA = CURATED / "eda"
OUT_FIG = REPORTS / "figures"

df = pd.read_parquet(OUT_FEATURES / "dataset_analitico_indicadores.parquet")
df_curado, umbrales = aplicar_transformaciones_curadas(df)
print(len(df.columns), "->", len(df_curado.columns), "columnas; filas:", len(df_curado))
assert len(df_curado) == 1997

df_curado.to_parquet(OUT_FEATURES / "dataset_analitico_curado.parquet", index=False)
df_curado.to_csv(OUT_FEATURES / "dataset_analitico_curado.csv", index=False, encoding="utf-8")
umbrales.to_csv(OUT_EDA / "resumen_outliers_estratificado.csv", index=False, encoding="utf-8")
umbrales.head(10)

## Cuántos outliers hay

In [ ]:
proc = df_curado[df_curado["tipo_elemento_analitico"] == "proceso"]
filas = []
for nombre in ["carga", "ingresos", "congestion", "duracion"]:
    leve = int(proc["es_outlier_" + nombre + "_iqr"].sum())
    severo = int(proc["es_outlier_severo_" + nombre + "_iqr"].sum())
    filas.append({"variable": nombre, "leves": leve, "severos": severo, "pct_leves": round(leve / len(proc) * 100, 1)})
conteo = pd.DataFrame(filas)
conteo

In [ ]:
conteo.set_index("variable")[["leves", "severos"]].plot(kind="bar", figsize=(8, 4), rot=0)
plt.title("Outliers detectados en " + str(len(proc)) + " procesos")
plt.ylabel("procesos")
plt.tight_layout()
plt.show()

## Las transformaciones no cambian el orden (Spearman = 1 si el orden se conserva)

In [ ]:
rho_log = spearmanr(proc["atendidas"], proc["log_atendidas"])[0]
con_tc = proc[proc["tasa_congestion"].notnull()]
rho_win = spearmanr(con_tc["tasa_congestion"], con_tc["tasa_congestion_winsorizada"])[0]
print("atendidas vs log_atendidas:", round(rho_log, 4))
print("tasa_congestion vs winsorizada:", round(rho_win, 4))

## Figura del reporte: antes y después

In [ ]:
materias = sorted(proc["materia_homologada"].unique())
etiquetas = [m[:18] for m in materias]
paneles = [
    (proc, "atendidas", "A. Carga atendida cruda"),
    (proc, "log_atendidas", "B. Carga atendida con log(1 + x)"),
    (con_tc, "tasa_congestion", "C. Tasa de congestión original"),
    (con_tc, "tasa_congestion_winsorizada", "D. Tasa de congestión winsorizada (P95 por estrato)"),
]
fig, ejes = plt.subplots(2, 2, figsize=(15, 12))
for k in range(4):
    datos, columna, titulo = paneles[k]
    eje = ejes[k // 2][k % 2]
    series = []
    for m in materias:
        series.append(datos[datos["materia_homologada"] == m][columna].values)
    eje.boxplot(series, tick_labels=etiquetas, showmeans=True)
    eje.set_title(titulo, fontweight="bold")
    eje.tick_params(axis="x", rotation=25)
    eje.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(OUT_FIG / "02_boxplots_outliers_iqr.png", dpi=200)
plt.show()